In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 Полный анализ проекта компьютерного зрения\n",
    "## Детектирование объектов для ассистивных систем\n",
    "\n",
    "**Студент:** [Ваше имя]\n",
    "**Дата:** $(document).ready(function(){$('#date').text(new Date().toLocaleDateString('ru-RU'))}) <span id='date'></span>"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 1. ИМПОРТ БИБЛИОТЕК\n",
    "# ============================================================\n",
    "\n",
    "import os\n",
    "import cv2\n",
    "import json\n",
    "import glob\n",
    "import torch\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from collections import Counter\n",
    "from PIL import Image\n",
    "\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "from configs.config import COCO_CLASSES_RU, DEVICE, NUM_CLASSES\n",
    "from src.models.faster_rcnn import get_model as get_faster\n",
    "from src.models.ssd import get_model as get_ssd\n",
    "from ultralytics import YOLO\n",
    "\n",
    "%matplotlib inline\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "print(\"✅ Все библиотеки загружены\")\n",
    "print(f\"✅ Устройство: {DEVICE}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "# ЧАСТЬ 1: АНАЛИЗ ДАТАСЕТА\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 2. СТАТИСТИКА ДАТАСЕТА\n",
    "# ============================================================\n",
    "\n",
    "def analyze_dataset(label_dir, name):\n",
    "    \"\"\"Анализирует датасет и возвращает статистику\"\"\"\n",
    "    class_counts = Counter()\n",
    "    total_objects = 0\n",
    "    files_with_objects = 0\n",
    "    total_files = 0\n",
    "    \n",
    "    for label_file in glob.glob(os.path.join(label_dir, \"*.txt\")):\n",
    "        total_files += 1\n",
    "        with open(label_file, 'r') as f:\n",
    "            lines = f.readlines()\n",
    "            if len(lines) > 0:\n",
    "                files_with_objects += 1\n",
    "                total_objects += len(lines)\n",
    "                for line in lines:\n",
    "                    cls = int(line.split()[0])\n",
    "                    class_counts[cls] += 1\n",
    "    \n",
    "    return {\n",
    "        'name': name,\n",
    "        'total_files': total_files,\n",
    "        'files_with_objects': files_with_objects,\n",
    "        'total_objects': total_objects,\n",
    "        'avg_objects': total_objects / total_files if total_files > 0 else 0,\n",
    "        'class_counts': class_counts\n",
    "    }\n",
    "\n",
    "# Пути к данным\n",
    "TRAIN_LABELS = \"../data/raw/labels/train2017\"\n",
    "VAL_LABELS = \"../data/raw/labels/val2017\"\n",
    "TRAIN_IMAGES = \"../data/raw/images/train2017\"\n",
    "VAL_IMAGES = \"../data/raw/images/val2017\"\n",
    "\n",
    "# Анализ\n",
    "train_stats = analyze_dataset(TRAIN_LABELS, 'Train')\n",
    "val_stats = analyze_dataset(VAL_LABELS, 'Val')\n",
    "\n",
    "# Вывод\n",
    "print(\"=\"*60)\n",
    "print(\"📊 СТАТИСТИКА ДАТАСЕТА\")\n",
    "print(\"=\"*60)\n",
    "for stats in [train_stats, val_stats]:\n",
    "    print(f\"\\n{stats['name']}:\")\n",
    "    print(f\"  Изображений: {stats['total_files']}\")\n",
    "    print(f\"  С объектами: {stats['files_with_objects']}\")\n",
    "    print(f\"  Всего объектов: {stats['total_objects']}\")\n",
    "    print(f\"  Среднее объектов/изобр: {stats['avg_objects']:.2f}\")\n",
    "\n",
    "# Сохраняем статистику для отчета\n",
    "stats_df = pd.DataFrame([\n",
    "    {'Датасет': 'Train', 'Изображения': train_stats['total_files'], \n",
    "     'Объекты': train_stats['total_objects'], 'Среднее': f\"{train_stats['avg_objects']:.2f}\"},\n",
    "    {'Датасет': 'Val', 'Изображения': val_stats['total_files'], \n",
    "     'Объекты': val_stats['total_objects'], 'Среднее': f\"{val_stats['avg_objects']:.2f}\"}\n",
    "])\n",
    "print(\"\\n\" + stats_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 3. РАСПРЕДЕЛЕНИЕ КЛАССОВ\n",
    "# ============================================================\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(16, 8))\n",
    "fig.suptitle('Распределение классов в датасете COCO', fontsize=16, fontweight='bold')\n",
    "\n",
    "for idx, stats in enumerate([train_stats, val_stats]):\n",
    "    ax = axes[idx]\n",
    "    classes = [COCO_CLASSES_RU[i] for i in stats['class_counts'].keys() if i < len(COCO_CLASSES_RU)]\n",
    "    counts = list(stats['class_counts'].values())[:len(classes)]\n",
    "    \n",
    "    sorted_idx = np.argsort(counts)[::-1][:15]\n",
    "    top_classes = [classes[i] for i in sorted_idx]\n",
    "    top_counts = [counts[i] for i in sorted_idx]\n",
    "    \n",
    "    bars = ax.barh(top_classes, top_counts, color=plt.cm.viridis(np.linspace(0, 1, len(top_classes))))\n",
    "    ax.set_xlabel('Количество объектов', fontsize=12)\n",
    "    ax.set_title(f'{stats[\"name\"]} датасет (топ-15)', fontsize=14, fontweight='bold')\n",
    "    ax.grid(True, alpha=0.3, axis='x')\n",
    "    \n",
    "    for bar, val in zip(bars, top_counts):\n",
    "        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2, \n",
    "                f'{val}', va='center', fontsize=8)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/plots/class_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "print(\"✅ Сохранено: class_distribution.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "# ЧАСТЬ 2: РЕЗУЛЬТАТЫ МОДЕЛЕЙ\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 4. МЕТРИКИ МОДЕЛЕЙ (из ваших CSV)\n",
    "# ============================================================\n",
    "\n",
    "data = {\n",
    "    'Модель': ['Faster R-CNN', 'SSD', 'YOLOv5', 'YOLOv8n', 'YOLOv8s'],\n",
    "    'Precision': [0.3253, 0.3975, 0.6243, 0.6632, 0.6636],\n",
    "    'Recall': [0.5193, 0.3481, 0.5829, 0.5276, 0.5994],\n",
    "    'F1-score': [0.4000, 0.3711, 0.6029, 0.5877, 0.6299],\n",
    "    'Mean IoU': [0.7182, 0.7040, 0.8286, 0.8641, 0.8826],\n",
    "    'TP': [188, 126, 211, 191, 217],\n",
    "    'FP': [390, 191, 127, 97, 110],\n",
    "    'FN': [174, 236, 151, 171, 145]\n",
    "}\n",
    "\n",
    "df = pd.DataFrame(data)\n",
    "df_sorted = df.sort_values('F1-score', ascending=False)\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА МЕТРИК\")\n",
    "print(\"=\"*60)\n",
    "display(df_sorted.style.background_gradient(subset=['Precision', 'Recall', 'F1-score', 'Mean IoU'], cmap='RdYlGn'))"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 5. ГРАФИКИ МЕТРИК\n",
    "# ============================================================\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "fig.suptitle('Сравнение моделей детектирования объектов', fontsize=16, fontweight='bold')\n",
    "\n",
    "# 5.1. Precision, Recall, F1\n",
    "ax1 = axes[0, 0]\n",
    "x = np.arange(len(df['Модель']))\n",
    "width = 0.25\n",
    "\n",
    "bars1 = ax1.bar(x - width, df['Precision'], width, label='Precision', color='#4CAF50')\n",
    "bars2 = ax1.bar(x, df['Recall'], width, label='Recall', color='#2196F3')\n",
    "bars3 = ax1.bar(x + width, df['F1-score'], width, label='F1-score', color='#FF9800')\n",
    "\n",
    "ax1.set_xlabel('Модель', fontsize=11)\n",
    "ax1.set_ylabel('Значение', fontsize=11)\n",
    "ax1.set_title('Precision, Recall и F1-score', fontsize=12, fontweight='bold')\n",
    "ax1.set_xticks(x)\n",
    "ax1.set_xticklabels(df['Модель'], rotation=45, ha='right')\n",
    "ax1.legend()\n",
    "ax1.set_ylim(0, 1)\n",
    "ax1.grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "# 5.2. Mean IoU\n",
    "ax2 = axes[0, 1]\n",
    "colors = ['#FF5722', '#FFC107', '#CDDC39', '#8BC34A', '#4CAF50']\n",
    "bars = ax2.bar(df['Модель'], df['Mean IoU'], color=colors)\n",
    "ax2.set_ylabel('Mean IoU', fontsize=11)\n",
    "ax2.set_title('Точность локализации (Mean IoU)', fontsize=12, fontweight='bold')\n",
    "ax2.set_xticklabels(df['Модель'], rotation=45, ha='right')\n",
    "ax2.set_ylim(0, 1)\n",
    "ax2.grid(True, alpha=0.3, axis='y')\n",
    "for bar, val in zip(bars, df['Mean IoU']):\n",
    "    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,\n",
    "            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')\n",
    "\n",
    "# 5.3. TP, FP, FN\n",
    "ax3 = axes[1, 0]\n",
    "x = np.arange(len(df['Модель']))\n",
    "width = 0.25\n",
    "\n",
    "bars1 = ax3.bar(x - width, df['TP'], width, label='TP', color='#4CAF50')\n",
    "bars2 = ax3.bar(x, df['FP'], width, label='FP', color='#FF5722')\n",
    "bars3 = ax3.bar(x + width, df['FN'], width, label='FN', color='#FFC107')\n",
    "\n",
    "ax3.set_xlabel('Модель', fontsize=11)\n",
    "ax3.set_ylabel('Количество', fontsize=11)\n",
    "ax3.set_title('TP, FP и FN', fontsize=12, fontweight='bold')\n",
    "ax3.set_xticks(x)\n",
    "ax3.set_xticklabels(df['Модель'], rotation=45, ha='right')\n",
    "ax3.legend()\n",
    "ax3.grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "# 5.4. Рейтинг по F1\n",
    "ax4 = axes[1, 1]\n",
    "df_sorted = df.sort_values('F1-score', ascending=True)\n",
    "colors = ['#4CAF50' if i == len(df_sorted)-1 else '#8BC34A' if i >= len(df_sorted)-2 else '#CDDC39' for i in range(len(df_sorted))]\n",
    "bars = ax4.barh(df_sorted['Модель'], df_sorted['F1-score'], color=colors)\n",
    "ax4.set_xlabel('F1-score', fontsize=11)\n",
    "ax4.set_title('Рейтинг моделей по F1-score', fontsize=12, fontweight='bold')\n",
    "ax4.set_xlim(0, 1)\n",
    "ax4.grid(True, alpha=0.3, axis='x')\n",
    "for bar, val in zip(bars, df_sorted['F1-score']):\n",
    "    ax4.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2.,\n",
    "            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/plots/all_metrics.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "print(\"✅ Сохранено: all_metrics.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "# ЧАСТЬ 3: ВИЗУАЛИЗАЦИЯ ПРЕДСКАЗАНИЙ\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 6. ЗАГРУЗКА МОДЕЛЕЙ И ВИЗУАЛИЗАЦИЯ\n",
    "# ============================================================\n",
    "\n",
    "def load_model(model_name):\n",
    "    \"\"\"Загружает модель по имени\"\"\"\n",
    "    if 'yolov8n' in model_name.lower():\n",
    "        return YOLO('../results/models/best_yolov8n.pt')\n",
    "    elif 'yolov8s' in model_name.lower():\n",
    "        return YOLO('../results/models/best_yolov8s.pt')\n",
    "    elif 'yolov5' in model_name.lower():\n",
    "        return YOLO('../results/models/best_yolov5.pt')\n",
    "    elif 'ssd' in model_name.lower():\n",
    "        model = get_ssd(NUM_CLASSES + 1)\n",
    "        model.load_state_dict(torch.load('../results/models/best_ssd.pth', map_location=DEVICE))\n",
    "        model.to(DEVICE).eval()\n",
    "        return model\n",
    "    elif 'faster' in model_name.lower():\n",
    "        model = get_faster(NUM_CLASSES + 1)\n",
    "        model.load_state_dict(torch.load('../results/models/best_faster.pth', map_location=DEVICE))\n",
    "        model.to(DEVICE).eval()\n",
    "        return model\n",
    "\n",
    "def predict(model, model_name, image_path):\n",
    "    \"\"\"Получает предсказания\"\"\"\n",
    "    image = cv2.imread(image_path)\n",
    "    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n",
    "    \n",
    "    if 'yolo' in model_name.lower():\n",
    "        results = model.predict(image_path, conf=0.3, verbose=False)[0]\n",
    "        if results.boxes is not None:\n",
    "            boxes = results.boxes.xyxy.cpu().numpy()\n",
    "            labels = results.boxes.cls.cpu().numpy().astype(int)\n",
    "            scores = results.boxes.conf.cpu().numpy()\n",
    "    else:\n",
    "        tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).float() / 255.0\n",
    "        tensor = tensor.to(DEVICE)\n",
    "        with torch.no_grad():\n",
    "            pred = model([tensor])[0]\n",
    "        boxes = pred['boxes'].cpu().numpy()\n",
    "        labels = pred['labels'].cpu().numpy()\n",
    "        scores = pred['scores'].cpu().numpy()\n",
    "        if 'ssd' in model_name.lower() or 'faster' in model_name.lower():\n",
    "            labels = labels - 1\n",
    "    \n",
    "    return boxes, labels, scores\n",
    "\n",
    "def draw_boxes(img, boxes, labels, scores, threshold=0.3):\n",
    "    \"\"\"Рисует боксы на изображении\"\"\"\n",
    "    img = img.copy()\n",
    "    for box, label, score in zip(boxes, labels, scores):\n",
    "        if score < threshold:\n",
    "            continue\n",
    "        x1, y1, x2, y2 = map(int, box)\n",
    "        class_name = COCO_CLASSES_RU[label] if label < len(COCO_CLASSES_RU) else f\"class_{label}\"\n",
    "        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)\n",
    "        cv2.putText(img, f'{class_name} {score:.2f}', (x1, max(20, y1-10)),\n",
    "                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)\n",
    "    return img"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Визуализация на 2-х изображениях\n",
    "test_images = [\n",
    "    '../data/raw/val2017/000000000139.jpg',\n",
    "    '../data/raw/val2017/000000000285.jpg'\n",
    "]\n",
    "models_to_show = ['YOLOv8s', 'YOLOv8n', 'SSD']\n",
    "\n",
    "fig, axes = plt.subplots(len(test_images), len(models_to_show) + 1, figsize=(18, 10))\n",
    "fig.suptitle('Сравнение предсказаний моделей', fontsize=16, fontweight='bold')\n",
    "\n",
    "loaded_models = {name: load_model(name) for name in models_to_show}\n",
    "\n",
    "for row, img_path in enumerate(test_images):\n",
    "    # Оригинал\n",
    "    img = cv2.imread(img_path)\n",
    "    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n",
    "    axes[row, 0].imshow(img_rgb)\n",
    "    axes[row, 0].axis('off')\n",
    "    axes[row, 0].set_title('Original', fontsize=10)\n",
    "    \n",
    "    for col, (name, model) in enumerate(loaded_models.items(), start=1):\n",
    "        boxes, labels, scores = predict(model, name, img_path)\n",
    "        result = draw_boxes(img_rgb.copy(), boxes, labels, scores)\n",
    "        axes[row, col].imshow(result)\n",
    "        axes[row, col].axis('off')\n",
    "        axes[row, col].set_title(f'{name}\\n({len(boxes)})', fontsize=10)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/plots/predictions_comparison.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "print(\"✅ Сохранено: predictions_comparison.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "# ЧАСТЬ 4: ВЫВОДЫ\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# 7. ИТОГОВЫЕ ВЫВОДЫ\n",
    "# ============================================================\n",
    "\n",
    "print(\"=\"*70)\n",
    "print(\"📊 ИТОГОВЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ\")\n",
    "print(\"=\"*70)\n",
    "\n",
    "# Лучшая модель\n",
    "best = df.loc[df['F1-score'].idxmax()]\n",
    "print(f\"\\n🥇 ЛУЧШАЯ МОДЕЛЬ: {best['Модель']}\")\n",
    "print(f\"   Precision: {best['Precision']:.4f}\")\n",
    "print(f\"   Recall:    {best['Recall']:.4f}\")\n",
    "print(f\"   F1-score:  {best['F1-score']:.4f}\")\n",
    "print(f\"   Mean IoU:  {best['Mean IoU']:.4f}\")\n",
    "\n",
    "# Худшая модель\n",
    "worst = df.loc[df['F1-score'].idxmin()]\n",
    "print(f\"\\n❌ ХУДШАЯ МОДЕЛЬ: {worst['Модель']}\")\n",
    "print(f\"   F1-score: {worst['F1-score']:.4f}\")\n",
    "\n",
    "# Рекомендация\n",
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"💡 РЕКОМЕНДАЦИИ\")\n",
    "print(\"=\"*70)\n",
    "print(\"\"\"\n",
    "1. YOLOv8s - лучший выбор для систем реального времени:\n",
    "   - Высокая точность (0.66) и полнота (0.60)\n",
    "   - Отличная локализация (IoU 0.88)\n",
    "   - Работает в реальном времени\n",
    "\n",
    "2. YOLOv8n - для мобильных устройств:\n",
    "   - Почти такая же точность как YOLOv8s\n",
    "   - Легкая и быстрая\n",
    "\n",
    "3. SSD и Faster R-CNN не рекомендуются:\n",
    "   - Требуют больше данных\n",
    "   - Сложнее в настройке\n",
    "   - Медленнее работают\n",
    "\"\"\")"
   ]
  }
 ]
}